# 📚 Preparación de Datos para RAG Multimodal (datos estructurados y no estructurados)

## ¿Qué vamos a hacer en este notebook?

Este es el **primer paso de todo proyecto RAG**: preparar los datos. Un LLM solo puede responder bien sobre *tus* documentos si antes los **ingerimos, troceamos y guardamos** en un formato que luego se pueda buscar.

Aquí construiremos una **base de conocimiento multimodal**, es decir, que mezcla **dos tipos de fuentes**:

| Tipo de dato | Ejemplo | Cómo lo procesamos |
|--------------|---------|--------------------|
| **No estructurado — texto** | PDFs | Extraemos el texto y lo troceamos en *chunks*. |
| **No estructurado — imágenes** | PNG/JPG | Pedimos a un LLM de visión que las **describa con palabras** (*verbalización*). |

Al final, todo (texto de PDFs + descripciones de imágenes) acaba en **una sola tabla** con la misma estructura, lista para vectorizar.

---

### 🗺️ ¿Dónde encaja esto dentro del flujo de RAG?

| Paso | Notebook | Qué hace |
|------|----------|----------|
| **1** | **`Data_Prep` (este)** | **Ingiere y trocea documentos e imágenes en una tabla.** |
| 2 | `Vector_Embeddings` | Convierte cada fragmento de texto en un vector. |
| 3 | `Vector_Index_Creation` | Guarda esos vectores en un índice buscable. |
| 4 | `RAG_Application` | Recupera los fragmentos relevantes y se los pasa al LLM. |
| 5 | `RAG_Evaluation` | Mide la calidad de las respuestas. |

> 📚 **Nota para el aprendizaje**: lee siempre las celdas de texto (como esta) antes de ejecutar el código de debajo. Explican el *qué* y, sobre todo, el *por qué*.

> 🛠️ **Nota sobre esta versión**: este notebook ha sido **revisado y mejorado** respecto a la versión original. A lo largo del recorrido encontrarás avisos **🔧 Mejora** que explican qué se corrigió y por qué (nombres de variables engañosos, un bug de IDs duplicados, librerías obsoletas, uso correcto de *Volumes*, etc.).

![Data_Prep](./Assets/Data_Prep.png)

## 🧠 Conceptos clave antes de empezar

### 🗂️ Volumes vs. DBFS (¡importante!)

Para guardar archivos (PDFs, imágenes...) en Databricks hay dos mecanismos, y **no son iguales**:

| | **Unity Catalog Volumes** ✅ (lo que usamos) | **DBFS** ⚠️ (heredado) |
|---|---|---|
| Ruta | `/Volumes/<catalogo>/<esquema>/<volumen>/...` | `/dbfs/...` o `dbfs:/...` |
| Gobierno | **Gobernado por Unity Catalog**: permisos, linaje, auditoría | Sin gobierno fino |
| Estado | **Recomendado actualmente** | **En desuso** (el *DBFS root* se considera *legacy*) |
| Uso | Almacenar datos no tabulares de forma segura y gobernada | Compatibilidad antigua |

> 🔧 **Mejora aplicada**: la versión original llamaba a sus variables `dbfs_target_folder`, `dbfs_subfolder`, etc., **pero las rutas eran de Volumes** (`/Volumes/...`). Ese nombre era **engañoso**. En esta versión las hemos renombrado a `volume_*` para que el código diga la verdad. **Regla práctica: para datos nuevos, usa siempre Volumes, no DBFS.**

### 🧩 Otros términos

| Concepto | ¿Qué es? |
|----------|----------|
| **Chunking** (*troceado*) | Partir un documento largo en fragmentos pequeños. Necesario porque los LLM y los buscadores trabajan mejor con trozos manejables. |
| **Chunk overlap** (*solapamiento*) | Repetir un poco de texto entre fragmentos contiguos para no cortar ideas a la mitad. |
| **Verbalización de imágenes** | Usar un LLM de visión para convertir una imagen en una **descripción de texto** que sí se puede vectorizar y buscar. |
| **Multimodal** | Que combina varias modalidades de datos (texto + imágenes) en un mismo sistema. |
| **Delta Table** | El formato de tabla de Databricks (transaccional, versionado) donde guardaremos los resultados. |

## 1️⃣ Instalación de librerías y utilidades

Instalamos solo lo que **realmente** usa este notebook:

| Librería | Para qué la usamos |
|----------|--------------------|
| `pypdf` | Leer y extraer el texto de los PDFs. |
| `langchain-text-splitters` | Trocear el texto (*chunking*) con `RecursiveCharacterTextSplitter`. |
| `numpy==1.26.4` | Dependencia de soporte; fijamos esta versión para evitar incompatibilidades de NumPy 2.x con Arrow/Spark al crear DataFrames. |

> 🔧 **Mejoras aplicadas en las dependencias:**
> - **`PyPDF2` → `pypdf`**: `PyPDF2` está **descontinuado** (deprecado). Su sucesor oficial es `pypdf`, con la **misma API** (`from pypdf import PdfReader`).
> - **Se añadió `langchain-text-splitters`**: el código importa `langchain_text_splitters`, que vive en **este** paquete. La versión original instalaba `langchain-community`, que no lo garantiza → riesgo de `ModuleNotFoundError`.
> - **Se eliminaron `openai` y `langchain-community`**: no se usan en este notebook. Menos dependencias = entornos más rápidos y reproducibles.

> 💡 **Nota**: las versiones exactas son un ejemplo razonable; si tu *Databricks Runtime* trae otras, ajústalas. Lo importante es **fijarlas** (`==`) para que el notebook sea reproducible.

In [ ]:
%pip install -q numpy==1.26.4 pypdf==5.1.0 langchain-text-splitters==0.3.4

## 2️⃣ Reiniciando el entorno de Python

Cuando instalas librerías nuevas con `%pip`, Python ya tenía cargada en memoria la versión anterior (o ninguna). Para que el notebook "vea" las librerías recién instaladas hay que **reiniciar el intérprete**.

`dbutils.library.restartPython()` hace justo eso, **sin perder la conexión con el cluster**.

> ⚠️ **Efecto importante**: al reiniciar se **borran todas las variables** que tuvieras en memoria. Por eso la regla de oro es: **instala → reinicia → y SOLO DESPUÉS empieza a programar.** Todo el código "de verdad" va después de esta celda.

In [ ]:
dbutils.library.restartPython()

## 3️⃣ Configuración: catálogo, esquema y *Volume*

Antes de tocar datos, definimos **en un solo sitio** los nombres que usaremos en todo el notebook (catálogo, esquema y volumen) y creamos los objetos en Unity Catalog **en el orden correcto**:

1. **Esquema** `RAG` (dentro del catálogo actual) → donde vivirán las tablas.
2. **Volume** `genai_lab` **dentro del esquema `RAG`** → donde guardaremos los archivos (PDFs e imágenes).

> 🔧 **Mejoras aplicadas:**
> - **Toda la configuración centralizada** en variables al principio (`catalog_name`, `schema_name`, `volume_name`, rutas). Antes el nombre del catálogo se calculaba en otra celda y el volumen estaba *hardcodeado* en el esquema `default`, separado de las tablas (que iban en `RAG`). Ahora **todo vive junto, en el esquema `RAG`**.
> - **Orden correcto**: creamos el esquema *antes* que el volumen y *antes* que las tablas. En la versión original el esquema se creaba mucho más tarde, y solo funcionaba porque el volumen caía en `default` (que siempre existe). Eso era frágil.
> - **Sin placeholders manuales**: usamos `current_catalog()` en lugar de `YOUR_UNITY_CATALOG_NAME`, así el notebook funciona sin editar a mano (siempre que tengas permisos en ese catálogo).

> 💡 **Para la certificación**: la jerarquía de Unity Catalog es **`catálogo → esquema → objeto`** (tabla, vista, *volume*, función, modelo...). Un *Volume* es un objeto de UC para datos **no tabulares** (archivos).

In [ ]:
# ------------------------------------------------------------------
# Configuración central del notebook (se define UNA sola vez aquí)
# ------------------------------------------------------------------
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
schema_name  = "RAG"
volume_name  = "genai_lab"

# Creamos esquema y volumen EN ORDEN (esquema primero, luego el volumen dentro de él)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

# Ruta base del Volume (gobernado por Unity Catalog) donde guardaremos los archivos.
# Nota: es una ruta de VOLUMES (/Volumes/...), NO de DBFS (/dbfs/...).
volume_base    = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/knowledge"
volume_docs    = f"{volume_base}/docs"
volume_images  = f"{volume_base}/images"

# Carpeta local (en el repo del workspace) que trae los archivos de ejemplo
source_folder  = "./knowledge"

print(f"Catálogo : {catalog_name}")
print(f"Esquema  : {schema_name}")
print(f"Volume   : {volume_name}")
print(f"Ruta base del Volume: {volume_base}")

### 📥 Copiando los archivos de ejemplo al Volume

Los archivos de ejemplo vienen en la carpeta local `./knowledge` (dentro del repo del workspace). Los **copiamos al Volume** para tener una **única fuente de verdad gobernada** desde la que procesar todo lo demás.

El código recorre las subcarpetas `docs` e `images` y copia cada archivo (con contenido) al Volume usando `dbutils.fs.cp`.

| Pieza | Qué hace |
|-------|----------|
| `dbutils.fs.mkdirs(ruta)` | Crea carpetas dentro del Volume. |
| `dbutils.fs.cp("file:<local>", "<volume>")` | Copia un archivo local al Volume. El prefijo `file:` indica que el origen está en el sistema de archivos **local** del driver. |
| `os.path.getsize(src) > 0` | Salta archivos vacíos (evita copiar basura). |

In [ ]:
import os

# Aseguramos que existe la carpeta base del Volume
dbutils.fs.mkdirs(volume_base)

# Mapa: subcarpeta local -> subcarpeta destino en el Volume
subfolders = {
    "docs":   volume_docs,
    "images": volume_images,
}

for subfolder, volume_subfolder in subfolders.items():
    local_subfolder = os.path.join(source_folder, subfolder)
    os.makedirs(local_subfolder, exist_ok=True)   # crea la carpeta local si no existe
    dbutils.fs.mkdirs(volume_subfolder)            # crea la carpeta en el Volume

    for filename in os.listdir(local_subfolder):
        src = os.path.join(local_subfolder, filename)
        dst = f"{volume_subfolder}/{filename}"
        # Copiamos solo archivos reales y con contenido (tamaño > 0)
        if os.path.isfile(src) and os.path.getsize(src) > 0:
            dbutils.fs.cp(f"file:{os.path.abspath(src)}", dst, recurse=False)
            print(f"Copiado: {filename} -> {dst}")

### ✅ Verificación 1: archivos en la carpeta local

Comprobamos que los archivos de ejemplo están en la carpeta local del repo (origen de la copia).

In [ ]:
print("Contenido de la carpeta local:")
print(os.listdir(source_folder))
print("\n images:")
print(os.listdir(os.path.join(source_folder, "images")))
print("\n docs:")
print(os.listdir(os.path.join(source_folder, "docs")))

### ✅ Verificación 2: archivos en el Volume de Unity Catalog

Comprobamos que la copia llegó bien al Volume gobernado (`/Volumes/...`). Este es el destino desde el que procesaremos los datos a partir de aquí.

In [ ]:
print("Contenido del Volume:")
print(dbutils.fs.ls(volume_base))
print("\n images:")
print(dbutils.fs.ls(volume_images))
print("\n docs:")
print(dbutils.fs.ls(volume_docs))

## 4️⃣ Procesando las imágenes (modalidad visual)

### Paso 4.1 — Tabla con las imágenes en Base64

Las imágenes son binarios; no se pueden vectorizar ni buscar tal cual. El plan es:

1. **Leer** las imágenes del Volume como binario.
2. **Codificarlas en Base64** (texto) para poder guardarlas en una columna y, después, reenviarlas al LLM de visión.

| Pieza | Qué hace |
|-------|----------|
| `spark.read.format("binaryFile")` | Lee archivos binarios (imágenes) como filas, con su `path` y `content`. |
| `base64(content)` | Convierte los bytes de la imagen en una cadena Base64. |
| `createOrReplaceTempView` | Crea una vista temporal para poder consultarla con SQL. |

> 🔧 **Mejora aplicada**: ahora leemos desde la variable `volume_images` (la **única fuente de verdad** en el Volume), en lugar de repetir la ruta `/Volumes/.../default/...` *hardcodeada*.

In [ ]:
# Leemos las imágenes del Volume como binario
images_df = spark.read.format("binaryFile").load(volume_images)
images_df.createOrReplaceTempView("images_temp")

# Guardamos una tabla con la ruta y la imagen codificada en Base64
spark.sql(f"""
CREATE OR REPLACE TABLE {schema_name}.images_metadata AS
SELECT
  path AS content_path,
  base64(content) AS base64_content
FROM images_temp
""")

display(spark.table(f"{schema_name}.images_metadata"))

### Paso 4.2 — Verbalización de imágenes con un LLM de visión

Aquí ocurre la magia de lo **multimodal**: usamos un **LLM con visión** (`databricks-llama-4-maverick`) para que **describa con palabras** cada imagen. Esa descripción (`chunk`) es texto, así que después podrá vectorizarse y buscarse igual que el texto de los PDFs.

| Pieza | Qué hace |
|-------|----------|
| `ai_query('<modelo>', '<prompt>', files => ...)` | Función SQL que llama a un modelo. El parámetro `files =>` le pasa la imagen al modelo de visión. |
| `unbase64(base64_content)` | Reconvierte el Base64 (texto) de vuelta a **bytes**, que es lo que el modelo necesita. |
| `chunk` | La columna de **texto** resultante: la descripción de la imagen. |

> 🔧 **Mejora aplicada**: el *prompt* original (*"what is this image about?"*) era demasiado vago y producía descripciones pobres. Lo hemos hecho **más rico y orientado a recuperación**: pedimos objetos, texto visible, datos y contexto, porque cuanto **más descriptivo** sea el `chunk`, **mejor lo encontrará** la búsqueda semántica después.

> 💡 **Para la certificación**: a esta técnica se la llama *image verbalization* o *captioning*. Es la forma estándar de meter imágenes en un pipeline de RAG basado en texto. `ai_query` permite invocar *Foundation Models* directamente desde SQL.

In [ ]:
# Prompt rico y orientado a recuperación: cuanto más detallada sea la descripción,
# mejor la encontrará luego la búsqueda semántica.
verbalization_prompt = (
    "Describe esta imagen de forma detallada y objetiva para un sistema de búsqueda. "
    "Incluye: objetos y personas presentes, cualquier texto visible (titulares, etiquetas, cifras), "
    "tipo de gráfico o diagrama y los datos que muestra, colores dominantes y el contexto o tema general."
)

spark.sql(f"""
CREATE OR REPLACE TABLE {schema_name}.images_verbalization AS
SELECT
  *,
  ai_query(
    'databricks-llama-4-maverick',
    '{verbalization_prompt}',
    files => unbase64(base64_content)
  ) AS chunk
FROM {schema_name}.images_metadata
""")

display(spark.table(f"{schema_name}.images_verbalization"))

## 5️⃣ Procesando los PDFs (modalidad texto) con *chunking*

Ahora la otra modalidad: el **texto** de los PDFs. El proceso es:

1. **Leer** cada PDF del Volume y extraer su texto (`pypdf`).
2. **Trocearlo** (*chunking*) en fragmentos con `RecursiveCharacterTextSplitter`.
3. Guardar cada fragmento como una fila con su `content_path` y su `chunk`.

### ✂️ ¿Por qué trocear? ¿Y qué es el solapamiento?

Un PDF puede tener miles de palabras, pero los buscadores semánticos y los LLM funcionan mejor con **trozos pequeños y enfocados**. El `RecursiveCharacterTextSplitter` corta el texto intentando respetar la estructura natural (primero por párrafos `\n\n`, luego líneas, luego frases...).

| Parámetro | Valor | Qué significa |
|-----------|-------|---------------|
| `chunk_size` | `2000` | Tamaño máximo de cada fragmento (en caracteres). |
| `chunk_overlap` | `400` | Caracteres que se **repiten** entre fragmentos contiguos para no cortar una idea por la mitad. |
| `separators` | `["\n\n", "\n", ". ", " ", ""]` | Lista de cortes preferidos, de más "limpio" a más "forzado". |

> ⚖️ **Compromiso del chunk size**: trozos **muy grandes** → más contexto pero recuperación menos precisa y más coste de tokens. Trozos **muy pequeños** → muy precisos pero pueden perder contexto. `2000/400` es un punto de partida razonable que **deberías ajustar** según tus documentos.

> 🔧 **Mejoras aplicadas:**
> - **Leemos los PDFs desde el Volume** (`volume_docs`), no desde la carpeta local. Así procesamos la **misma fuente gobernada** que el resto del pipeline (antes mezclaba: copiaba al Volume pero leía del local).
> - **`PyPDF2` → `pypdf`** (la librería moderna; `PyPDF2` está deprecado).
> - **Se eliminó el import sin usar** `from langchain_core.documents import Document`.
> - **`chunk_overlap` 500 → 400**: 500 sobre 2000 era un 25% de solapamiento (mucha duplicación y coste); 400 (20%) es más equilibrado.
> - **Se quitó la variable `i` no usada** del bucle de fragmentos.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader


def perform_fixed_size_chunking(document, chunk_size=2000, chunk_overlap=400):
    """
    Trocea un texto en fragmentos con solapamiento.
    Usa RecursiveCharacterTextSplitter, que intenta cortar por los separadores
    más "limpios" primero (párrafos, líneas, frases...).
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return text_splitter.split_text(document)


# Leemos los PDFs directamente desde el Volume (única fuente de verdad gobernada).
# Las rutas de Volumes (/Volumes/...) se pueden abrir con open() normal de Python.
all_docs = []

for filename in os.listdir(volume_docs):
    file_path = f"{volume_docs}/{filename}"
    if not filename.lower().endswith(".pdf"):
        continue

    with open(file_path, "rb") as f:
        reader = PdfReader(f)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text

    if text.strip():
        for chunk in perform_fixed_size_chunking(text):
            if chunk.strip():
                all_docs.append({
                    "content_path": file_path,   # ruta en el Volume
                    "chunk": chunk,
                })

if all_docs:
    df = spark.createDataFrame(all_docs)
    print(f"Total de fragmentos creados: {df.count()}")
    print("\nFragmentos por documento:")
    df.groupBy("content_path").count().show(truncate=False)
    display(df)
else:
    print("No se extrajo ningún fragmento de los documentos.")

### Paso 5.1 — Guardando los fragmentos de PDF como tabla Delta

Persistimos el DataFrame de fragmentos en una tabla Delta (`RAG.docs_chunks`). Usamos `mode("overwrite")` para que la celda sea **idempotente**: puedes re-ejecutarla y siempre obtienes el mismo resultado limpio, sin duplicar filas.

In [ ]:
df.write.mode("overwrite").saveAsTable(f"{schema_name}.docs_chunks")

## 6️⃣ Creando la tabla final multimodal de RAG

Último paso: **unimos** las dos modalidades en una sola tabla (`RAG.final_rag_dataset`) con la misma estructura:

| Columna | Qué es |
|---------|--------|
| `id` | Identificador **único** de cada fragmento (será la *primary key* del índice vectorial). |
| `content_path` | De dónde salió (ruta del PDF o de la imagen en el Volume). |
| `chunk` | El texto: un trozo de PDF **o** la descripción de una imagen. |

Combinamos con `UNION ALL` las descripciones de imágenes (`images_verbalization`) y los fragmentos de PDF (`docs_chunks`).

> 🐞 **Bug corregido (¡el más importante de este notebook!)**
>
> La versión original generaba el `id` con `monotonically_increasing_id()` **por separado en cada rama** del `UNION ALL`:
> ```sql
> SELECT monotonically_increasing_id() AS id, ... FROM images_verbalization
> UNION ALL
> SELECT monotonically_increasing_id() AS id, ... FROM docs_chunks
> ```
> El problema: `monotonically_increasing_id()` reinicia su secuencia en cada consulta/partición, así que **las dos ramas pueden generar los mismos IDs** → **claves duplicadas**. Y el `Vector_Index_Creation` usa `primary_key="id"`, que **exige IDs únicos** (filas con id repetido se pisarían o el índice fallaría).
>
> ✅ **Solución**: primero hacemos el `UNION ALL` y **después** asignamos el `id` **una sola vez** sobre el conjunto ya unido. Así la unicidad está garantizada.

> 💡 **Alternativa aún más robusta**: usar `uuid()` en lugar de `monotonically_increasing_id()` genera identificadores únicos globales y estables, útil si más adelante haces *upserts* o cargas incrementales.

In [ ]:
# Generamos el 'id' UNA sola vez sobre el conjunto YA unido -> IDs únicos garantizados
spark.sql(f"""
CREATE OR REPLACE TABLE {schema_name}.final_rag_dataset AS
SELECT
  monotonically_increasing_id() AS id,
  content_path,
  chunk
FROM (
  SELECT content_path, chunk FROM {schema_name}.images_verbalization
  UNION ALL
  SELECT content_path, chunk FROM {schema_name}.docs_chunks
)
""")

display(spark.table(f"{schema_name}.final_rag_dataset"))

# Verificación rápida: nº total de filas == nº de IDs distintos (no hay duplicados)
spark.sql(f"""
SELECT COUNT(*) AS total_filas, COUNT(DISTINCT id) AS ids_unicos
FROM {schema_name}.final_rag_dataset
""").show()

## ✅ Resumen y siguientes pasos

### Lo que has construido
Una **base de conocimiento multimodal** en una sola tabla Delta (`RAG.final_rag_dataset`) con columnas `id`, `content_path` y `chunk`, combinando:
- 📄 **Texto de PDFs** troceado en *chunks*.
- 🖼️ **Imágenes verbalizadas** (descritas con un LLM de visión).

### 🔧 Mejoras aplicadas en esta versión (repaso)
| Problema original | Solución |
|-------------------|----------|
| Variables `dbfs_*` apuntando a *Volumes* (nombres engañosos) | Renombradas a `volume_*` + tabla DBFS vs Volumes |
| `monotonically_increasing_id()` por rama → IDs duplicados | `id` generado una vez tras el `UNION ALL` (+ chequeo de unicidad) |
| PDFs leídos del local pero copiados al Volume | Todo se procesa desde el Volume (fuente única) |
| Volume en esquema `default`, tablas en `RAG` | Todo unificado en el esquema `RAG` |
| `PyPDF2` deprecado + import sin usar | `pypdf` y limpieza de imports |
| Dependencias innecesarias / faltantes | Solo `pypdf`, `langchain-text-splitters`, `numpy` |
| Config dispersa y placeholders manuales | Configuración central con `current_catalog()` |
| Esquema creado tarde | Esquema y volumen creados al principio, en orden |

### 🧠 Puntos típicos de certificación
- **Volumes** (gobernados por UC) son el estándar actual para archivos; **DBFS root** es *legacy*.
- Jerarquía de UC: **catálogo → esquema → objeto**.
- `ai_query(...)` invoca *Foundation Models* desde SQL (texto **y** visión con `files =>`).
- La *primary key* de un índice vectorial **debe ser única**.
- *Chunking* + *overlap*: equilibrio entre precisión de recuperación y conservación de contexto.

### ➡️ Siguiente notebook: `Vector_Embeddings`
Ya tenemos los datos limpios y troceados. El siguiente paso es **convertir cada `chunk` en un vector** para poder buscarlos por significado. 🚀